# Micrograd

A clean implementation of a tiny autograd engine, following the concepts from Karpathy's micrograd.

In [ ]:
import math
from graphviz import Digraph

In [2]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self._backward = lambda: None

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        out = Value(
            self.data + other.data,
            (self, other),
            _op='+'
        )

        def _backward():
            self.grad += out.grad * 1.0
            other.grad += out.grad * 1.0

        out._backward = _backward
        return out

    def __mul__(self, other):
        out = Value(
            self.data * other.data,
            (self, other),
            _op='*'
        )

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)

        out = Value(
            t,
            (self,),
            _op='tanh'
        )

        def _backward():
            self.grad += (1 - out.data**2) * out.grad

        out._backward = _backward
        return out

## Test the computational graph

In [3]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)

d = a * b + c

print(d)
print(d._prev)
print(d._op)
print(hasattr(d, "_backward"))

Value(data=4.0)
{Value(data=-6.0), Value(data=10.0)}
+
True


In [4]:
# Test tanh
x = Value(0.8813735870195432)
o = x.tanh()

print(o)
print(o.__dict__)
print(hasattr(o, "_backward"))

Value(data=0.7071067811865476)
{'data': 0.7071067811865476, 'grad': 0.0, '_prev': {Value(data=0.8813735870195432)}, '_op': 'tanh', '_backward': <function Value.tanh.<locals>._backward at 0x7fde5960a2a0>}
True


## Neuron-style example

In [5]:
x1 = Value(2.0)
x2 = Value(0.0)

w1 = Value(-3.0)
w2 = Value(1.0)

b = Value(6.8813735870195432)

x1w1 = x1 * w1
x2w2 = x2 * w2

x1w1x2w2 = x1w1 + x2w2
n = x1w1x2w2 + b

o = n.tanh()

print(o)
print(o.__dict__)

Value(data=0.7071067811865476)
{'data': 0.7071067811865476, 'grad': 0.0, '_prev': {Value(data=0.8813735870195432)}, '_op': 'tanh', '_backward': <function Value.tanh.<locals>._backward at 0x7fde5960aac0>}


In [8]:


def trace(root):
    nodes = set()
    edges = set()

    def build(v):
        if v not in nodes:
            nodes.add(v)

            for child in v._prev:
                edges.add((child, v))
                build(child)

    build(root)
    return nodes, edges


def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})

    nodes, edges = trace(root)

    for n in nodes:
        uid = str(id(n))

        # Value node
        dot.node(
            name=uid,
            label="{ data %.4f | grad %.4f }" % (n.data, n.grad),
            shape='record'
        )

        # Operation node
        if n._op:
            dot.node(
                name=uid + n._op,
                label=n._op
            )

            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [7]:
print(o._backward())

None


In [9]:
draw_dot(d)

NameError: name 'Digraph' is not defined